# Topic 8: Graphs — BFS, DFS & Applications

## 8.1 What Is a Graph?

A graph is a collection of **nodes (vertices)** connected by **edges**. Trees are a special type of graph (connected, acyclic). Graphs model relationships — social networks, road maps, web links, neural network connections, knowledge bases.

### 📊 [VISUAL] Graph Types and Representations

```
GRAPH TYPES:

UNDIRECTED (edges have no direction):    DIRECTED (edges have direction):
  A -- B -- C                             A --> B --> C
  |         |                             ^           |
  D ------- E                             |           v
                                          D <--------- E

WEIGHTED: edges have a cost/distance
  A --3-- B --2-- C   (travel 3 miles A->B, 2 miles B->C)

REPRESENTATIONS:
  1. Adjacency List (preferred):  {A:[B,D], B:[A,C,E], ...}
     Space: O(V+E)   Best for sparse graphs (most real-world)

  2. Adjacency Matrix: 2D grid, matrix[i][j]=1 if edge exists
     Space: O(V²)    Best for dense graphs or O(1) edge lookup
```

---

## 8.2 Graph Implementation and Traversals

In [ ]:
from collections import defaultdict, deque


class Graph:
    """Undirected graph using adjacency list."""

    def __init__(self):
        self.adj = defaultdict(list)   # {node: [neighbours]}

    def add_edge(self, u, v, directed=False):
        self.adj[u].append(v)
        if not directed:
            self.adj[v].append(u)

    # ── BFS — Level-order, shortest path (unweighted) ────────────────────────
    def bfs(self, start):
        visited = {start}
        queue   = deque([start])
        order   = []
        while queue:
            node = queue.popleft()
            order.append(node)
            for nb in sorted(self.adj[node]):   # sorted for reproducibility
                if nb not in visited:
                    visited.add(nb)
                    queue.append(nb)
        return order

    # ── DFS — Depth-first (recursive) ────────────────────────────────────────
    def dfs(self, start, visited=None):
        if visited is None: visited = set()
        visited.add(start)
        result = [start]
        for nb in sorted(self.adj[start]):
            if nb not in visited:
                result += self.dfs(nb, visited)
        return result

    # ── BFS SHORTEST PATH ─────────────────────────────────────────────────────
    def shortest_path(self, start, end):
        if start == end: return [start]
        visited = {start}
        queue   = deque([[start]])
        while queue:
            path = queue.popleft()
            node = path[-1]
            for nb in self.adj[node]:
                if nb == end:
                    return path + [nb]
                if nb not in visited:
                    visited.add(nb)
                    queue.append(path + [nb])
        return []   # no path found

    # ── DETECT CYCLE (directed) — DFS with 3-colour marking ──────────────────
    def has_cycle_directed(self):
        WHITE, GRAY, BLACK = 0, 1, 2
        colour = defaultdict(int)

        def dfs(node):
            colour[node] = GRAY
            for nb in self.adj[node]:
                if colour[nb] == GRAY:   return True   # back edge = cycle!
                if colour[nb] == WHITE:
                    if dfs(nb):          return True
            colour[node] = BLACK
            return False

        return any(dfs(n) for n in self.adj if colour[n] == WHITE)


g = Graph()
for u, v in [('A','B'), ('A','C'), ('B','D'), ('C','D'), ('D','E')]:
    g.add_edge(u, v)

print("BFS from A:",              g.bfs('A'))               # A B C D E
print("DFS from A:",              g.dfs('A'))               # A B D C E
print("Shortest path A->E:",      g.shortest_path('A','E')) # ['A','B','D','E']
print("Has cycle (undirected)?",  g.has_cycle_directed())   # False

## 8.3 Topological Sort — DAG Processing Order

In [ ]:
# Topological sort: order nodes so all edges go left->right
# ONLY works on DAGs (Directed Acyclic Graphs)
# Use case: task scheduling, ML pipeline order, build systems

def topological_sort(graph_adj):
    from collections import defaultdict, deque
    in_degree = defaultdict(int)
    all_nodes = set(graph_adj.keys())
    for node in graph_adj:
        for nb in graph_adj[node]:
            in_degree[nb] += 1
            all_nodes.add(nb)

    queue  = deque(n for n in all_nodes if in_degree[n] == 0)
    result = []
    while queue:
        node = queue.popleft()
        result.append(node)
        for nb in graph_adj.get(node, []):
            in_degree[nb] -= 1
            if in_degree[nb] == 0:
                queue.append(nb)

    if len(result) != len(all_nodes):
        raise ValueError('Graph has a cycle — topological sort impossible')
    return result


# ML pipeline: each step depends on the previous
pipeline = {
    'load_data':   ['preprocess'],
    'preprocess':  ['feature_eng', 'split'],
    'feature_eng': ['train'],
    'split':       ['train'],
    'train':       ['evaluate'],
    'evaluate':    [],
}
order = topological_sort(pipeline)
print("ML Pipeline execution order:")
for i, step in enumerate(order, 1):
    print(f"  Step {i}: {step}")

## 8.4 Count Islands — Classic DFS Grid Problem

In [ ]:
# Classic interview problem: count connected components in a 2D grid
def count_islands(grid):
    """
    Count number of islands (connected groups of 1s) in a 2D grid.
    Time: O(rows × cols),  Space: O(rows × cols) for recursion stack.
    """
    if not grid: return 0
    rows, cols = len(grid), len(grid[0])
    count = 0

    def dfs(r, c):
        if r < 0 or r >= rows or c < 0 or c >= cols: return
        if grid[r][c] != 1: return
        grid[r][c] = '#'    # mark visited (avoid extra visited set)
        dfs(r+1, c); dfs(r-1, c)
        dfs(r, c+1); dfs(r, c-1)

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 1:
                dfs(r, c)
                count += 1
    return count


grid1 = [
    [1, 1, 0, 0, 0],
    [1, 1, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 0, 1, 1],
]
print("Islands in grid1:", count_islands(grid1))   # 3

grid2 = [
    [1, 1, 0],
    [0, 1, 0],
    [0, 0, 1],
]
print("Islands in grid2:", count_islands(grid2))   # 2

> 🔵 **[AI/ML]** Graphs are everywhere in modern AI:
> - **Graph Neural Networks (GNNs):** molecular property prediction (atoms=nodes, bonds=edges), fraud detection.
> - **Knowledge Graphs:** facts as `(entity, relation, entity)` triples — used in RAG for LLMs.
> - **ML Pipeline DAG:** Airflow, Kubeflow, MLflow model pipelines as DAGs — `topological_sort()` determines execution order.
> - **Transformer attention** = complete bipartite graph where every token attends to every other token → O(n²) complexity.

---

## ✏️ Exercises — Graphs

**[EXERCISE 8.1 — Medium]** Write `count_islands(grid)` (exercise above already shown — now do it without modifying the input grid, using a separate visited set).

**[EXERCISE 8.2 — Advanced]** Implement **Dijkstra's algorithm** for shortest path in a weighted graph. Input: `{node: [(neighbour, weight), ...]}` and start node. Return dict of shortest distances from start to all nodes. Use `heapq`.


----